# Thread Runs vs Stateless Runs in LangGraph

当你通过 LangGraph API（SDK 或 Remote Graph）调用 graph 时，有两种执行模式：

- **Thread Run** — 有状态，绑定到一个 thread
- **Stateless Run** — 无状态，一次性执行

## Thread Run（有状态）

Thread 是一个**持久化的对话容器**。每次 run 结束后，graph 的 state 会被保存为 checkpoint。下一次 run 会从上次的 checkpoint 继续。

### 工作流程：
1. 创建一个 thread
2. 在这个 thread 上执行 graph（run）
3. State 被保存
4. 下一次 run 自动加载之前的 state

### 特点：
- ✅ 保存 checkpoint（state 持久化）
- ✅ 支持多轮对话（记住历史）
- ✅ 支持 Human-in-the-loop（interrupt → 修改 state → resume）
- ✅ 支持 Time travel（回到之前的 checkpoint）
- ⚠️ 需要数据库存储 checkpoint（Postgres/SQLite）

In [ ]:
from langgraph_sdk import get_client
from langchain_core.messages import HumanMessage

client = get_client(url="http://localhost:8123")

# 1. 创建 thread
thread = await client.threads.create()
print(f"Thread ID: {thread['thread_id']}")

# 2. 第一轮对话 — state 会被保存
result = await client.runs.wait(
    thread["thread_id"],
    assistant_id="task_maistro",
    input={"messages": [{"role": "human", "content": "Add a task: finish LangGraph course"}]}
)
print("Run 1 done")

# 3. 第二轮对话 — 同一个 thread，graph 记得之前的对话
result = await client.runs.wait(
    thread["thread_id"],
    assistant_id="task_maistro",
    input={"messages": [{"role": "human", "content": "What tasks do I have?"}]}
)
print("Run 2 done — graph remembers the previous task")

## Stateless Run（无状态）

不绑定任何 thread。每次执行都是独立的，不保存 state，执行完就丢弃。

### 工作流程：
1. 直接执行 graph（不需要 thread）
2. 得到结果
3. State 被丢弃，不保存任何东西

### 特点：
- ✅ 轻量，无需数据库
- ✅ 适合一次性任务
- ✅ 高并发友好（无锁竞争）
- ❌ 不保存 checkpoint
- ❌ 不支持多轮对话
- ❌ 不支持 interrupt/resume
- ❌ 不支持 time travel

In [ ]:
# Stateless run — 传入 thread_id=None
result = await client.runs.wait(
    None,  # 没有 thread！
    assistant_id="task_maistro",
    input={"messages": [{"role": "human", "content": "What is 2+2?"}]}
)
print("Stateless run done — no state saved anywhere")

## 类比

| | Thread Run | Stateless Run |
|---|---|---|
| 类比 | 微信聊天窗口（有历史记录） | 匿名提问箱（问完就忘） |
| 数据库 | 需要（存 checkpoint） | 不需要 |
| 多轮对话 | ✅ | ❌ |
| Human-in-the-loop | ✅ | ❌ |
| Time travel | ✅ | ❌ |
| 性能 | 较重（读写 DB） | 轻量 |
| 典型场景 | 聊天机器人、agent、需要记忆的任务 | 单次问答、数据处理、批量任务 |

## 什么时候用哪个？

**用 Thread Run：**
- 需要记住之前的对话
- 需要暂停等待人类反馈（human-in-the-loop）
- 需要回溯到之前的状态（debugging / time travel）
- 构建聊天机器人或多步 agent

**用 Stateless Run：**
- 一次性处理，不需要历史
- 高并发批量处理（每个请求独立）
- 简单的 transform/pipeline 任务
- 不想管理 thread 生命周期

## 本地 graph 的对应关系

在本地使用 graph 时：

```python
# 相当于 Thread Run — 传入 checkpointer + thread_id
graph = builder.compile(checkpointer=MemorySaver())
result = graph.invoke(input, config={"configurable": {"thread_id": "abc"}})

# 相当于 Stateless Run — 不传 checkpointer
graph = builder.compile()  # 没有 checkpointer
result = graph.invoke(input)  # 没有 thread_id
```

部署到 LangGraph server 后，这个区别变成了：有没有传 `thread_id`。